In [1]:
from pathlib import Path
import numpy as np
from PIL import Image
from collections import defaultdict
import pandas as pd

base_path = Path("/kaggle/input/vesuvius-challenge-ink-detection/")
train_path = base_path / "train"
test_path = base_path / "test"



In [2]:
ink_ratios = []
for train_frag in train_path.iterdir():
    if not train_frag.is_dir():
        continue
    ink_file = train_frag / "inklabels.png"
    mask_file = train_frag / "mask.png"
    if not ink_file.is_file() or not mask_file.is_file():
        continue
    ink_arr = np.array(Image.open(ink_file).convert("L"))
    ink_bin = (ink_arr > 0).astype(np.uint8)
    mask_arr = np.array(Image.open(mask_file).convert("L"))
    mask_bin = (mask_arr > 0).astype(np.uint8)
    mask_count = mask_bin.sum()
    if mask_count == 0:
        continue
    ink_ratios.append(ink_bin.sum() / mask_count)

avg_ink_ratio = float(np.mean(ink_ratios)) if ink_ratios else 0.01
print(f"Average ink‑to‑mask ratio from training data: {avg_ink_ratio:.4f}")

pred_images = []
fragment_names = []

for test_fragment in test_path.iterdir():
    if not test_fragment.is_dir():
        continue
    mask_file = test_fragment / "mask.png"
    if not mask_file.exists():
        continue

    mask_arr = np.array(Image.open(mask_file).convert("L"))
    mask_bin = (mask_arr > 0).astype(np.uint8)

    n_mask = mask_bin.sum()
    n_pred = max(1, int(avg_ink_ratio * n_mask))
    # Use flat indices of mask pixels (corrected)
    pos_indices = np.flatnonzero(mask_bin)
    pred_idx = pos_indices[:n_pred]  # first indices, deterministic

    pred_mask = np.zeros_like(mask_bin)
    pred_mask.flat[pred_idx] = 1

    pred_images.append(pred_mask)
    fragment_names.append(test_fragment.name)
    print(
        f"Processed fragment '{test_fragment.name}': mask {mask_bin.shape}, "
        f"predicted {n_pred} ink pixels."
    )

print(f"Total fragments processed: {len(pred_images)}")




/usr/local/lib/python3.11/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (140973980 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


Average ink‑to‑mask ratio from training data: 0.1777
Processed fragment 'a': mask (7606, 5249), predicted 4453110 ink pixels.
Total fragments processed: 1


In [3]:
def rle(img: np.ndarray) -> str:
    """
    Convert a binary mask to run‑length encoding (row‑major order, 1‑indexed).
    This implementation correctly handles runs that start at the first pixel
    or end at the last pixel.
    """
    flat = img.flatten(order="C")
    flat = np.where(flat > 0, 1, 0).astype(np.uint8)

    changes = np.where(np.diff(np.concatenate([[0], flat, [0]])) != 0)[0] + 1
    starts = changes[0::2]
    ends = changes[1::2]
    lengths = ends - starts

    rle_pairs = np.column_stack((starts, lengths)).flatten()
    return " ".join(map(str, rle_pairs))




In [4]:
submission = defaultdict(list)

for idx, fragment_name in enumerate(fragment_names):
    submission["Id"].append(fragment_name)
    submission["Predicted"].append(rle(pred_images[idx]))

output_path = Path("/kaggle/working/submission.csv")
pd.DataFrame.from_dict(submission).to_csv(output_path, index=False)
print(f"Submission written to {output_path}")

Submission written to /kaggle/working/submission.csv
